# Destination access and network measures (GHSCI)
## Mexicali Urban Liveability Index — `WP01_ghsci_access_network`

**Lead:** Carl Higgs
**Indicators assigned:** 39
**Schema version:** 1.0.0

Pedestrian-network destination analysis run through the Global Healthy and Sustainable City Indicators software: distance to closest, access within threshold distance(s), and where applicable counts per km² and per 1,000 persons. Also includes the composites produced by the core GHSCI workflow -- the walkability index and its components (street connectivity, population density, access to daily living amenities) and the level of cycling traffic stress network.

Produced by the GHSCI pipeline for the Mexicali study region and exported to the ULI schema, so that every other work package has a worked reference implementation to follow. These are the only composites constructed within a work package; every other indicator is delivered as measured, and any further compositing happens centrally at the index step. The walkability index will be tailored to the arid context, mediated by the WP02 thermal comfort outputs.

> New to this project? Work through
> [`00_overview_and_schema.ipynb`](00_overview_and_schema.ipynb)
> first — it carries one indicator end to end. Then read
> [`docs/analyst_guide.md`](../docs/analyst_guide.md).

## How to work through this notebook

For each indicator assigned to you, in this order:

1. **Read the brief.** It reproduces everything the team already
   recorded in the workbook — the draft rationale, the article the
   indicator was adapted from, candidate data sources, and the open
   questions colleagues raised. Do not retype any of it; it is
   already in your metadata stub.
2. **Write the causal pathway sentence** (guide §2.1) and find
   **independent health evidence** for it (§2.2). Do this *before*
   looking for data. Fill in `meta['rationale']`.
3. **Find and document the data** (§3): citation, URL, date
   retrieved, licence, and whether it reaches Condesa.
4. **Compute** at the finest scale your data genuinely support.
   Produce a `DataFrame` with `geo_id` and `value`.
5. **Harmonise** with `uli.harmonise(...)`, label with
   `uli.label(...)`, and **deliver** with
   `uli.write_indicator(...)`.
6. **Look at the map.** Most errors are obvious in ten seconds and
   invisible in a table.

`uli.write_indicator` validates first and refuses to publish a
failing deliverable. While you are still iterating, pass
`allow_failure=True` to write a draft anyway.

Full guidance: [`docs/analyst_guide.md`](../docs/analyst_guide.md).
Schema: [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md).

## Framing the indicator against health evidence

Every indicator must be justified by evidence of a **meaningful
health or wellbeing benefit**, independent of the liveability
article it was adapted from. Those articles establish that an
indicator is used; they rarely establish that it matters.

Complete this sentence before you compute anything:

> *[what I measure]* changes *[a mechanism]*, which changes *[a
> behaviour or exposure]*, which affects *[a health outcome]*.

For most indicators in this project the behaviour is **walking for
transport**, **walking or recreation in public space**, or
**social contact** — and the exposure is **heat**, **air
pollution** or **injury risk**. Say which, using the vocabulary in
`uli.vocab.HEALTH_PATHWAYS`.

Prefer meta-analyses and systematic reviews, then reputable
guidance (WHO, UN-Habitat, PAHO, Secretaría de Salud), then cohort
studies and natural experiments. Record the **effect size with its
uncertainty**.

**If the evidence supports a different threshold from the one the
workbook proposes, use the evidence-based threshold** and say so in
`threshold_justification`. That is explicitly what the project
wants.

**Mexicali is arid and extremely hot.** Most of this literature
comes from temperate cities. Where the transfer is doubtful — for
example, distance-based walkability thresholds in a city where
summer maxima exceed 45 °C and shade rather than distance is the
binding constraint — record it in `rationale.arid_context`. That is
a contribution, not a caveat.

## When several workbook rows are really one indicator

The workbook harvested indicators article by article, so a single
construct sometimes appears as several rows seen through different
lenses or over different time periods. Air quality is the clearest
case:

| Row | What it is | Lens | Time basis |
|---|---|---|---|
| #292 Air quality | the index value itself | `quality` | `annual_mean` |
| #8 Good air quality | that value against a standard | `quality` | `threshold_share` |
| #293 Days with good air quality | how often the standard is met | `quantity` | `threshold_compliance_days` |
| #173 Days PM2.5 over WHO | the same, for one pollutant | `quantity` | `threshold_exceedance_days` |

These are not four indicators — they are one construct measured
four ways, and computing them separately would mean four
inconsistent methods and four sets of data documentation.

Deliver them as a **measure family**: give every measure the same
`measure_family` slug, and distinguish them with `temporal_basis`
(see `uli.vocab.TEMPORAL_BASES`) and `threshold`. They can still
live under separate workbook ids — the family slug is what tells
the index step, and Reimagina Urbana, that they belong together.

```python
for meta in (meta_292, meta_8, meta_293, meta_173):
    for measure in meta['measures']:
        measure['measure_family'] = 'air_quality'
    meta['data_sources'] = SHARED_SOURCES   # one method, one source
```

The same pattern applies to mean summer temperature versus days
above a comfort threshold (WP02), and to flood extent versus annual
average days of flooding (WP05).

## Condesa coverage is a requirement, not a nicety

The Condesa new development in south-east Mexicali is a project
focus area, and it defeats the usual assumptions:

- about **20%** of it falls outside the previously configured
  study region boundary;
- only **44%** of its area is covered by census manzana polygons,
  so a **manzana-native calculation reaches 33 of the 40
  fraccionamientos, while a `grid_100m`-native one reaches all
  40**;
- it is platted and roaded (43 km of street network in OpenStreetMap
  across 27 of the 40 fraccionamientos) but essentially unbuilt —
  **zero destinations**, and satellite-derived population products
  see almost nobody there.

**One thing is your decision: the native scale.** If your data
allow it, compute on the 100 m grid. That is the difference between
reaching all of Condesa and quietly missing a fifth of it.

Everything else is handled downstream. Population denominators,
the 2030 occupancy scenario and population-weighted exposure
statistics are a reporting-step concern (`uli.exposure`), decided
once for the whole project rather than by each analyst. Urban
fabric and exposure measures — land cover, air quality, heat,
hazards, street infrastructure — are properties of *place*, and
should be computed as such; who lives there is applied later.

Two things to record, though:

- `data_sources[].condesa_coverage` — whether your **source**
  reaches Condesa. Satellite imagery and OSM generally do; a 2020
  census variable or a household survey generally does not.
- `method.condesa_treatment` — what you did about it. Where a
  source does not reach Condesa, mark those rows `no_data` rather
  than omitting them.

The validator treats poor Condesa coverage as an **error**.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

        WORK_PACKAGE = 'WP01_ghsci_access_network'
        NOTEBOOK = 'notebooks/01_ghsci_access_network.ipynb'

            ## Assigned indicators (39)

            | # | Indicator | Code | Lenses |
            |---|---|---|---|
            | 46 | Access to community centres | `access_to_community_centres` | proximity, accessibility, quantity, density |
| 394 | Access to youth centres | `access_to_youth_centres` | proximity, accessibility, quantity, density |
| 11 | Access to bakery | `access_to_bakery` | proximity, accessibility, quantity, density |
| 129 | Access to markets | `access_to_markets` | proximity, accessibility, quantity, density |
| 132 | Access to minimarts | `access_to_minimarts` | proximity, accessibility, quantity, density |
| 169 | Access to petrol station (>= 250m away) | `access_to_petrol_station_250m_away` | proximity, accessibility, quantity, density |
| 232 | Access to restaurants | `access_to_restaurants` | proximity, accessibility, quantity, density |
| 367 | Access to supermarket | `access_to_supermarket` | proximity, accessibility, quantity, density |
| 58 | Access to cultural centers | `access_to_cultural_centers` | proximity, accessibility, quantity, density |
| 121 | Access to libraries | `access_to_libraries` | proximity, accessibility, quantity, density |
| 134 | Access to museums | `access_to_museums` | proximity, accessibility, quantity, density |
| 228 | Access to religious buildings | `access_to_religious_buildings` | proximity, accessibility, quantity, density |
| 340 | Population density | `population_density` | density |
| 97 | Access to health centres | `access_to_health_centres` | proximity, accessibility, quantity, density |
| 101 | Access to hospitals | `access_to_hospitals` | proximity, accessibility, quantity, density |
| 170 | Access to pharmacies | `access_to_pharmacies` | proximity, accessibility, quantity, density |
| 127 | Access to manufacturing district / centre | `access_to_manufacturing_district_centre` | proximity, accessibility, quantity, density |
| 74 | Walkability Index | `walkability_index` | quality |
| 91 | Access to public green space | `access_to_public_green_space` | proximity, accessibility, quantity, density |
| 162 | Access to parks (local, neighbourhood, district, regional) | `access_to_parks_local_neighbourhood_district_regional` | proximity, accessibility, quantity, density |
| 187 | Access to public open space | `access_to_public_open_space` | proximity, accessibility, quantity, density, quality |
| 69 | Access to fire station | `access_to_fire_station` | proximity, accessibility, quantity, density |
| 174 | Access to police station | `access_to_police_station` | proximity, accessibility, quantity, density |
| 218 | Co-location of recreational areas with public transport | `co_location_of_recreational_areas_with_public_transport` | proximity, accessibility, quantity, density, quality |
| 225 | Access to recreational areas | `access_to_recreational_areas` | proximity, accessibility, quantity, density |
| 257 | Access to sports centres | `access_to_sports_centres` | proximity, accessibility, quantity, density |
| 96 | Access to hawker/stall | `access_to_hawker_stall` | proximity, accessibility, quantity, density |
| 26 | Access to buses: stops | `access_to_buses_stops` | proximity, accessibility, quantity, density |
| 223 | Access to public transport | `access_to_public_transport` | proximity, accessibility, quantity, density |
| 3 | Access to activity centres | `access_to_activity_centres` | proximity, accessibility, quantity, density |
| 38 | Access to city center | `access_to_city_center` | proximity, accessibility, quantity, density |
| 125 | Access to major road | `access_to_major_road` | proximity, accessibility, quantity, density |
| 126 | Access to major road intersection | `access_to_major_road_intersection` | proximity, accessibility, quantity, density |
| 180 | Access to schools: primary | `access_to_schools_primary` | proximity, accessibility, quantity, density |
| 246 | Access to schools: secondary | `access_to_schools_secondary` | proximity, accessibility, quantity, density |
| 362 | Street connectivity | `street_connectivity` | quality |
| 379 | Access to schools: tertiary (universities) | `access_to_schools_tertiary_universities` | proximity, accessibility, quantity, density |
| 62 | Level of cycling traffic stress | `level_of_cycling_traffic_stress` | quality |
| 10 | Access to blue space | `access_to_blue_space` | proximity, accessibility, quantity, density |

            This work package has too many indicators to give each its
            own cell, and they share a single method, so they are
            processed as a batch. The per-indicator briefs are still
            available from the register:

            ```python
            uli.register.get(91)
            ```

In [ ]:
assigned = uli.register.load()
assigned = assigned[assigned['work_package'] == WORK_PACKAGE]
assigned[['indicator_id', 'indicator', 'indicator_code',
          'lenses', 'is_composite', 'pragmatic_method']]

### Batch processing

Build one metadata stub per indicator, complete the shared
documentation programmatically where it genuinely is
shared (method, software, data sources), and the
indicator-specific parts individually. Evidence is **not**
shared: each destination type needs its own health
citation, because the evidence for access to health
services is not the evidence for access to a bakery.

In [ ]:
metas = {
    int(row.indicator_id): uli.metadata_stub(
        int(row.indicator_id), analyst=ANALYST
    )
    for row in assigned.itertuples()
}

SHARED_SOURCES = [
    # {
    #     'name': 'OpenStreetMap',
    #     'custodian': 'OpenStreetMap contributors',
    #     'citation': 'OpenStreetMap contributors (2026). '
    #                 'Planet dump, Geofabrik extract Mexico.',
    #     'url': 'https://download.geofabrik.de/',
    #     'date_retrieved': '2026-04-10',
    #     'licence': 'ODbL-1.0',
    #     'redistributable': True,
    #     'spatial_resolution': 'vector',
    #     'temporal_coverage': '2026',
    #     'condesa_coverage': 'full',
    # },
]
for meta in metas.values():
    if SHARED_SOURCES:
        meta['data_sources'] = list(SHARED_SOURCES)
    meta['method']['software'] = ['GHSCI', 'OSMnx', 'pandana']
    meta['method']['notebook'] = NOTEBOOK

len(metas)

In [ ]:
# Calculation ---------------------------------------------------
# Produce, for each indicator, a native-scale DataFrame with
# geo_id and value, then harmonise and label it.
deliverables = {}

# for indicator_id, meta in metas.items():
#     native = ...
#     harmonised = uli.harmonise(native, NATIVE_SCALE,
#                                method=METHOD)
#     deliverables[indicator_id] = uli.label(
#         harmonised, meta,
#         measure_id=meta['measures'][0]['id'])

len(deliverables)

In [ ]:
# Validate and deliver ------------------------------------------
for indicator_id, results in deliverables.items():
    uli.write_indicator(results, metas[indicator_id])

---
## Check what this work package has delivered

In [ ]:
delivered, catalogue = uli.collect()
if len(catalogue):
    display(catalogue)
    print(delivered.groupby(['indicator_code', 'geo_level']).size())
else:
    print('Nothing delivered yet.')

In [ ]:
# Sanity-check a delivered measure on a map before you call it done.
# MEASURE = 'your_indicator_code__quantity'
# LEVEL = 'manzana'
# units = uli.geography.load(LEVEL).merge(
#     delivered.query('measure_id == @MEASURE and geo_level == @LEVEL'),
#     on='geo_id', how='left')
# ax = units.plot(column='value', legend=True, figsize=(11, 8),
#                 missing_kwds={'color': 'lightgrey'})
# condesa = uli.geography.load('condesa_fraccionamiento')
# condesa.boundary.plot(ax=ax, color='red', linewidth=1)
# ax.set_title(MEASURE)
# ax.set_axis_off()